In [ ]:
import jax
jax.config.update('jax_enable_x64', True)  # Configure precision before other JAX imports.
import jax.numpy as jnp
import jax_md
import numpy as np

from ase.io import read
from jax_md import units
from typing import Dict

from so3lr import to_jax_md
from so3lr import So3lrPotential

import time

# Nose-Hoover NPT

The setup below follows `npt_jaxmd.ipynb`. Refer to that notebook for a step-by-step explanation of the NPT integrator and periodic-coordinate conventions.

In [ ]:
# Simulation parameters

timestep_ps = 0.0005  # Time step in ps
npt_cycles = 25  # Number of NPT output cycles.
npt_steps = 5  # NPT steps per cycle; total steps are npt_cycles * npt_steps.

T_init_kelvin = 300  # Initial and target temperature in K.
pressure_bar = 1.01325  # Target pressure in bar.

chain = 3  # Number of chains in the Nose-Hoover chain.
chain_steps = 2  # Number of steps per chain.
sy_steps = 3

thermo = 100  # Thermostat value in the Nose-Hoover chain. 
baro = 1000  # Barostat value in the Nose-Hoover chain. 
hdf5_buffer_size = 3  # Frames per normal HDF5 flush.

# Default nose hoover chain parameters.
def default_nhc_kwargs(
    tau: jnp.float32, 
    overrides: Dict
) -> Dict:
    
    default_kwargs = {
        'chain_length': 3, 
        'chain_steps': 2, 
        'sy_steps': 3,
        'tau': tau
    }
    
    if overrides is None:
        return default_kwargs
  
    return {
      k: overrides.get(k, default_kwargs[k]) for k in default_kwargs
    }

In [ ]:
# Read some molecular structure. Here we read the water box from the test directory.
path_to_xyz = '../tests/test_data/water_64.xyz'
atoms = read(path_to_xyz, index=-1)
atoms.wrap()

# Repeat in each direction.
atoms = atoms * [2, 2, 2]

species = jnp.array(atoms.get_atomic_numbers())
masses = jnp.array(atoms.get_masses())
num_atoms = len(species)

# Use three box lengths: JAX-MD interprets a one-element vector as a
# one-dimensional volume, which is not valid for this 3D NPT simulation.
cell = np.asarray(atoms.get_cell())
np.testing.assert_allclose(cell, np.diag(np.diag(cell)))
box = jnp.asarray(np.diag(cell), dtype=jnp.float64)
assert np.all(np.asarray(box) > 0)

fractional_coordinates = True
displacement, shift = jax_md.space.periodic_general(box=box, fractional_coordinates=fractional_coordinates)

positions_init = jnp.asarray(
    atoms.get_scaled_positions(wrap=True), dtype=jnp.float64
)

model_name = 'so3lr-m-2606'
neighbor_fn, neighbor_fn_lr, energy_fn = to_jax_md(
    potential=So3lrPotential(
        model=model_name,
        dtype=jnp.float64  # or float32 for single precision
    ),
    displacement_or_metric=displacement,
    box_size=box,
    species=species,
    capacity_multiplier=1.25,
    buffer_size_multiplier_sr=1.25,
    buffer_size_multiplier_lr=1.25,
    minimum_cell_size_multiplier_sr=1.0,
    disable_cell_list=False,
    fractional_coordinates=fractional_coordinates
)

# Energy function.
energy_fn = jax.jit(energy_fn)
force_fn = jax.jit(jax_md.quantity.force(energy_fn))

# Initialize the short and long-range neighbor lists.
nbrs = neighbor_fn.allocate(
    positions_init,
    box=box
)
nbrs_lr = neighbor_fn_lr.allocate(
    positions_init,
    box=box
)

# For repeated execution of cell, delete the fire_state first, otherwise jit in jupyter environment 
# can meddle with things.
try:
    del fire_state
except NameError:
    pass

min_cycles = 5
min_steps = 10

fire_init, fire_apply = jax_md.minimize.fire_descent(
    energy_fn, 
    shift, 
    dt_start = 0.05, 
    dt_max = 0.1, 
    n_min = 2
)

fire_apply = jax.jit(fire_apply)
fire_state = fire_init(
    positions_init, 
    box=box, 
    neighbor=nbrs.idx,
    neighbor_lr=nbrs_lr.idx
)

@jax.jit
def step_fire_fn(i, fire_state):
    
    fire_state, nbrs, nbrs_lr = fire_state
    
    fire_state = fire_apply(
        fire_state, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx, 
        box=box
    )
    
    nbrs = nbrs.update(
        fire_state.position,
        neighbor=nbrs.idx,
        box=box
    )
    
    nbrs_lr = nbrs_lr.update(
        fire_state.position,
        neighbor_lr=nbrs_lr.idx,
        box=box
    )
    
    return fire_state, nbrs, nbrs_lr

print('Step\tE\tFmax')
print('--------------------------')
for i in range(min_cycles):
    fire_state, nbrs, nbrs_lr = jax.lax.fori_loop(
        0, 
        min_steps, 
        step_fire_fn, 
        (fire_state, nbrs, nbrs_lr)
    )

    # Reject a minimization chunk computed with an overflowed neighbor list.
    if bool(np.asarray(nbrs.did_buffer_overflow)) or bool(
        np.asarray(nbrs_lr.did_buffer_overflow)
    ):
        raise RuntimeError(
            'Neighbor-list capacity overflowed; increase the capacity or buffer multipliers.'
        )
    
    E = energy_fn(
        fire_state.position, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx,
        box=box
    )

    F = force_fn(
        fire_state.position, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx,
        box=box
    )
    
    print('{}\t{:.2f}\t{:.2f}'.format(i, E, np.abs(F).max()))

# Dictionary with the NHC settings.
new_nhc_kwargs = {
    'chain_length': chain, 
    'chain_steps': chain_steps, 
    'sy_steps': sy_steps
}

# Convert to metal unit system.
unit = units.metal_unit_system()

timestep = timestep_ps * unit['time']
T_init = T_init_kelvin * unit['temperature']
pressure = pressure_bar * unit['pressure']

rng_key = jax.random.PRNGKey(0)

# Choose Nose-Hoover thermostat.
init_fn, apply_fn = jax_md.simulate.npt_nose_hoover(
    energy_fn, 
    shift, 
    dt=timestep,
    pressure=pressure, 
    kT=T_init,
    barostat_kwargs=default_nhc_kwargs(baro * timestep, new_nhc_kwargs),
    thermostat_kwargs=default_nhc_kwargs(thermo * timestep, new_nhc_kwargs)
)

apply_fn = jax.jit(apply_fn)
init_fn = jax.jit(init_fn)

# Initialize the state from the relaxed positions and neighbor lists.
state = init_fn(
    rng_key, 
    fire_state.position, 
    box=box, 
    neighbor=nbrs.idx, 
    neighbor_lr=nbrs_lr.idx,
    kT=T_init,
    mass=masses
)

@jax.jit
def step_npt_fn(i, state):
    state, nbrs, nbrs_lr, box = state
    
    state = apply_fn(
        state,
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx, 
        kT=T_init,
        pressure=pressure
    )
    
    box = jax_md.simulate.npt_box(state)
    
    nbrs = nbrs.update(
        state.position, 
        neighbor=nbrs.idx, 
        box=box
    )
    
    nbrs_lr = nbrs_lr.update(
        state.position,
        neighbor_lr=nbrs_lr.idx,
        box=box
    )
    
    return state, nbrs, nbrs_lr, box

# Production run
Production runs need efficient trajectory and observable storage. Repeated text-format writes can dominate the cost for large systems, so this example buffers frames and appends them to HDF5.

This example uses SO3LR's maintained `init_hdf5_store` and `write_to_hdf5` helpers. Values are collected in an in-memory write buffer and flushed together for efficient I/O, but the buffer is not an extra dimension in the file. Datasets are frame-major: positions and momenta have shape `(n_frames, num_atoms, 3)`, and box has shape `(n_frames, 3)`. `batch_size` controls the HDF5 chunk length and the usual number of frames per flush. For details on the box representation in `jax_md`, see `npt_jaxmd.ipynb`.

In [ ]:
from so3lr.cli.so3lr_md import init_hdf5_store, write_to_hdf5
from so3lr import __version__

In [ ]:
# Initialize the frame-major HDF5 store.
hdf5_store = init_hdf5_store(
    save_to='hdf5/trajectory.hdf5',
    batch_size=hdf5_buffer_size,
    num_atoms=num_atoms,
    num_box_entries=3,
    exist_ok=False,
    observables=[],
    so3lr_version=__version__,
    ensemble='npt',
    md_dt_fs=1000 * timestep_ps,
    md_steps=npt_steps,
    atomic_numbers=np.asarray(species),
    cell_a0=cell,
    input_file=path_to_xyz,
    model_path=model_name,
    dtype=np.asarray(state.position).dtype,
    fractional_coordinates=False,
)

In [ ]:
# Track total time and step times averaged over cycle.
total_time = time.time()

momenta, positions, boxes = [], [], []
temperatures_md = []
print('Step\tKE\tPE\tTot. Energy\tTemp.\ttime/steps')
print('------------------------------------------------------------------------------------------------------------')
for i in range(npt_cycles):

    old_time = time.time()
    
    # Do `npt_steps` NPT steps.
    new_state, nbrs, nbrs_lr, new_box = jax.block_until_ready(
        jax.lax.fori_loop(
            0, 
            npt_steps, 
            step_npt_fn, 
            (state, nbrs, nbrs_lr, box)
        )
    )
    
    new_time = time.time()
    
    # A chunk computed with an overflowed neighbor list is not valid.
    if bool(np.asarray(nbrs.did_buffer_overflow)) or bool(
        np.asarray(nbrs_lr.did_buffer_overflow)
    ):
        raise RuntimeError(
            'Neighbor-list capacity overflowed; increase the capacity or buffer multipliers.'
        )
    state = new_state
    box = new_box

    # Save only a fully valid integration chunk.
    momenta.append(state.momentum)
    positions.append(jax_md.space.transform(box=box, R=state.position))
    boxes.append(box)
    
    # Flush each complete buffer. A final partial buffer is flushed below.
    if len(positions) >= hdf5_buffer_size:
        momenta, positions, boxes, _ = write_to_hdf5(hdf5_store, momenta, positions, boxes)
    

    # Calculate some quantities for printing
    KE = jax_md.quantity.kinetic_energy(
        momentum=state.momentum,
        mass=state.mass
    )
    
    PE = energy_fn(
        state.position,
        neighbor=nbrs.idx,
        neighbor_lr=nbrs_lr.idx, 
        box=box
    )
    
    T = jax_md.quantity.temperature(
        momentum=state.momentum,
        mass = state.mass
    ) / unit['temperature']
    temperatures_md.append(float(T))

    print(
        f'{(i+1)*npt_steps}\t{KE:.2f}\t{PE:.2f}\t{KE+PE:.3f}\t{T:.1f}\t{(new_time - old_time) / npt_steps:.2f}'
    )

# Never discard a final partial buffer. HDF5Store closes the file after append.
momenta, positions, boxes, _ = write_to_hdf5(hdf5_store, momenta, positions, boxes)
assert np.isfinite(np.asarray(temperatures_md)).all()
target_kelvin = T_init / unit['temperature']
assert 0.1 * target_kelvin < np.median(temperatures_md) < 10 * target_kelvin

print('Total_time: ', time.time() - total_time)

# Clear all caches
jax.clear_caches()

# Postprocess the Trajectory

In [ ]:
import h5py

with h5py.File('hdf5/trajectory.hdf5', mode='r') as file:
    trajectory_positions = np.array(file['positions'])
    trajectory_momenta = np.array(file['momenta'])
    trajectory_boxes = np.array(file['box'])

expected_vector_shape = (npt_cycles, num_atoms, 3)
assert trajectory_positions.shape == expected_vector_shape
assert trajectory_momenta.shape == expected_vector_shape
assert trajectory_boxes.shape == (npt_cycles, 3)
assert trajectory_positions.dtype == np.asarray(state.position).dtype
assert np.isfinite(trajectory_positions).all()
assert np.isfinite(trajectory_momenta).all()
assert np.isfinite(trajectory_boxes).all()
assert np.all(trajectory_boxes > 0)
print(trajectory_positions.shape)  # (n_frames, num_atoms, 3)